In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [17]:
os.chdir('/Users/quinnmackay/Documents/GitHub/BICC_Holocene_LayerCount/data/raw_lcs/Counts for Matchmaker')

In [20]:
#load LCs

edml_lc = pd.read_csv('EDML_LayerCount.txt', sep='\t')
edml_lc = edml_lc[edml_lc['Age'] <= 11999] #only look at the top 1000 m of the core, where we have tiepoints
wdc_lc = pd.read_csv('WDC_LayerCount.txt', sep='\t')

#load tiepoints
match_name = 'EDML-WDC'
core1=match_name.split('-')[0]
core2=match_name.split('-')[1]

big_table = pd.read_csv('/Users/quinnmackay/Documents/GitHub/IceTiepoint_Analysis/Network Analysis/big_table/all_tiepoints_big_table.csv')

compare = big_table[[f'{match_name}_{core1}', f'{match_name}_{core2}']].dropna()

#drop any rows with tiepoints lower than LC max depths
compare = compare[compare[f'{match_name}_{core1}'] <= edml_lc['Depth'].max()]
compare = compare[compare[f'{match_name}_{core2}'] <= wdc_lc['Depth'].max()]

compare.rename(columns={f'{match_name}_{core1}': f'{core1} m', f'{match_name}_{core2}': f'{core2} m'}, inplace=True)
compare.sort_values(by=[f'{core1} m'], inplace=True)
compare.reset_index(drop=True, inplace=True)

/var/folders/pq/tk8k94gn22v6mr07byx05hzc0000gn/T/ipykernel_81514/187597453.py:12: DtypeWarning: Columns (2,3,6,7,10,11,14,15,18,19,22,23,26,27,30,31,34,35,38,39,42,43,46,47,50,51,54,55,58,59,62,63,66,67,70,71,74,75,78,79,82,83,86,87,90,91,94,95,98,99,102,103,106,107,110,111,114,115,118,119,122,123,126,127,130,131,134,135,138,139,142,143,146,147,150,151,154,155,158,159,162,163) have mixed types. Specify dtype option on import or set low_memory=False.
  big_table = pd.read_csv('/Users/quinnmackay/Documents/GitHub/IceTiepoint_Analysis/Network Analysis/big_table/all_tiepoints_big_table.csv')


In [4]:
#get the ages from the tiepoints
compare[f'{core1} age'] = np.interp(compare[f'{core1} m'], edml_lc['Depth'], edml_lc['Age'])
compare[f'{core2} age'] = np.interp(compare[f'{core2} m'], wdc_lc['Depth'], wdc_lc['Age'])

#calc discrepancies
compare[f'difference ({core1}-{core2})'] = compare[f'{core1} age'] - compare[f'{core2} age'] # NEGATIVE means EDML undercounted relative
compare

#calc section error

compare[f'section error'] = compare[f'difference ({core1}-{core2})'].diff()
compare.loc[0, 'section error'] = 0
compare.to_excel('EDML-WDC_tiepoint_comparison.xlsx', index=False)